# Preserving and Resetting Object State

In this lesson, you will learn to choose which object fields survive saving and explain how temporary fields begin after restoration.

CSC-239 · Module 11 · Lesson 3 of 3

A campus activity program saves a person's score while treating the current session's view count as temporary. You will compare the original and restored objects, then transfer the same distinction to an equipment card.

Build on the previous lesson's supported object state, checked read, and cast. Every complete fixture defines its class and creates, uses, and removes its own private file. The [Module 11 glossary](terms.md) collects the terms explained in the reading.


## Learning Goals

- Choose saved and temporary fields, then predict and verify restored values and primitive defaults.
- Explain why restoration does not replay these serializable constructors or change original state.
- Build and test EquipmentCard across changed saved values and unexpected stored types.


## Why This Matters

Maya has seven points in a campus activity. Her name and point total are lasting facts that the program must recover after saving. A view count belongs to the current session: this small example sets it to one when constructing a card, without implementing a screen-view counter.

The report must recover the same owner and points, show the original view count, and show how the restored temporary count begins. Points and views are whole-number counts with different meanings. The program must also reject an unexpected restored type before calling a card method.

Applications often combine lasting records with temporary selections or observations. Choosing what to save keeps those roles clear. An excluded field still needs a suitable value when the object is restored; omitting it does not automatically recompute a useful replacement.


## Check Your Starting Point

Recall the object and file rules needed before choosing which fields to exclude.


Explain instance versus static state, object versus reference, and why the previous reader checked the restored type before casting. Recall when its streams closed and its private file was removed.


In [ ]:
Instance and static state:


Object and reference:


Check before cast:


Closing and cleanup:



<details>
<summary>Show answer</summary>

An instance field belongs to an individual object. A static field belongs to the class and is not saved as each object's ordinary instance state. A reference can identify an object without copying it.

The previous reader held the readObject result as Object, checked its actual non-null type with instanceof, and cast only within the matching branch. The cast allowed access through the expected type; it did not construct a new object. The writer closed before the reader opened, and both streams closed before finally deleted the example's own file.

The class definition and stored representation must agree. A fixed serialVersionUID participates in compatibility checking but does not make arbitrary class changes compatible. Use only the controlled files created by these examples.

</details>


## Video Demonstration

Watch the original score card retain its temporary count while a restored card receives the saved facts and a default temporary value. The connected reading below explains the same mechanism.

<video controls preload="metadata" width="960" style="max-width:100%;height:auto;">
<source src="media/03_preserving_and_resetting_object_state/demo.mp4" type="video/mp4">
<track kind="captions" src="media/03_preserving_and_resetting_object_state/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the object-state video transcript](media/03_preserving_and_resetting_object_state/transcript.md).


## Concept

### Separate saved facts from temporary session state

A campus activity program gives Maya a score card with seven points. The owner and points should survive saving because they describe the score. The program also records a temporary view count for the current session. Our small model initializes that count to one when a card is constructed; it does not implement an automatic counter that notices every screen view.

The previous lesson saved an object's supported state. This lesson asks which parts belong in that saved state. A **transient instance field** is a field excluded from default object serialization. The keyword **`transient`** makes that choice explicit:

```java
    private String owner;
    private int points;
    private transient int views;
```

These declarations belong inside ScoreCard. `owner` and `points` are ordinary instance fields included in this class's default saved state. `views` remains an int field of each object, but transient excludes its value from that default representation. All three fields are private; their different saving behavior comes from transient, not from private access.

The constructor still initializes the original object's fields normally:

```java
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
```

The parameters supply the saved facts, while the final assignment sets the session's initial view count. Marking views transient does not prevent ordinary assignment, stop the getter from reading it, or clear it when `writeObject` runs. The original card still has its view count after it is saved.

This choice is useful for temporary observations or state that should be recomputed for a new session. It requires a plan for what a restored object should use instead. Excluding a value is not the same as automatically recalculating a useful replacement. The next section explains exactly what happens in these ordinary serializable classes.

### Distinguish normal construction from default restoration

**Default restoration** reconstructs the serialized state of these ordinary Serializable classes without replaying their constructors or instance field initializers. The saved owner and points are restored from the representation. A transient field has no saved value in that representation, so it begins with its type's default value unless additional restoration behavior supplies something else.

A **default primitive value** is the initial value supplied for a primitive field before application-specific initialization. An int field's default is zero; a boolean field's default is false. This rule concerns fields, not an uninitialized local variable that Java would require you to assign before use.

Our original ScoreCard was constructed with `new ScoreCard("Maya", 7)`, so its constructor assigned views to one. Restoring its default serialized representation does not run that ScoreCard constructor again. The restored views field therefore begins at zero, even though the constructor contains `this.views = 1`.

The complete example compares both objects after the guarded read and cast:

```java
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
```

This fragment assumes the restored value passed the ScoreCard check from the previous lesson's reading pattern. The first line reports the saved facts as `Maya: 7`. The second reports `Original views: 1`, showing that saving did not clear the original object's field. The third reports `Restored views: 0`, showing the restored object's default temporary state.

The two objects now have equal saved facts but different view counts. That difference follows from the chosen serialization rule; it does not mean the stored points were lost. A reader that expects the restored view count to equal one would be assuming that the serializable constructor ran during restoration.

Keep this explanation scoped to the ordinary Serializable classes and default mechanism used here. Java has additional serialization mechanisms and inheritance rules; this lesson does not ask you to implement custom restoration. Within this example, the important decision is whether each temporary field's default is suitable for a freshly restored object.

The later examples use other temporary fields to test the same reasoning. Check the original value, identify whether the field is included, and then explain the restored value from that choice. Do not infer that all omitted state is safe to ignore merely because deserialization completed without an exception.

### Follow both score cards

The reading has established Maya, seven points, and the original view count of one. This animation follows which state is written and which value appears only when the new restored card is initialized.

<details class="animation-panel" open>
<summary>Show or hide animation: Compare saved facts and temporary views</summary>

<img src="media/03_preserving_and_resetting_object_state/saved_and_transient_state.gif" alt="Maya and seven points survive restoration; original views stays one and restored transient views begins zero." width="960" style="max-width:100%;height:auto;">

</details>

[View still: Compare saved facts and temporary views](media/03_preserving_and_resetting_object_state/saved_and_transient_state_still.png).

The original and restored objects are distinct. Owner and points survive in the representation; transient views is omitted. Original views remains one while restored views begins at zero. The complete runnable fixture appears just below the second comparison.

The loop lasts about 12.5 seconds. Use the summary control to hide or show motion; the still and explanation remain available.


### Apply the same rule to a boolean

SelectionCard keeps the label kit as lasting text and treats selected as temporary. Its constructor assigns true to the original selection. The complete reading example uses the same private file, closed writer/reader, checked cast, and cleanup pattern.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class SelectionCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    private transient boolean selected;
    public SelectionCard(String label) {
        this.label = label;
        this.selected = true;
    }
    public String getLabel() { return label; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("selection-card-", ".bin");
try {
    SelectionCard original = new SelectionCard("kit");
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof SelectionCard) {
            SelectionCard restored = (SelectionCard) value;
            System.out.println(restored.getLabel());
            System.out.println("Original: " + original.isSelected());
            System.out.println("Restored: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Its output is kit, Original: true, and Restored: false. The isSelected getter reads the boolean; it does not change it. The class constructor is not replayed during default restoration, so the excluded boolean begins at false.

<details class="animation-panel" open>
<summary>Show or hide animation: Compare construction with restored defaults</summary>

<img src="media/03_preserving_and_resetting_object_state/constructor_and_restored_defaults.gif" alt="Construction sets selected true, but restoration does not replay that assignment and the excluded boolean begins false." width="960" style="max-width:100%;height:auto;">

</details>

[View still: Compare construction with restored defaults](media/03_preserving_and_resetting_object_state/constructor_and_restored_defaults_still.png).

The constructor affects the object created with new. The restored object receives the saved label and a default temporary selection. Reading it does not change the original selection. This is a controlled comparison, not a custom restoration method.

The loop lasts about 12.5 seconds. Use the summary control to hide or show motion; the still and explanation remain available.


## Worked Example

### Define lasting facts and temporary state

The five imports supply the familiar object input/output streams, Serializable marker, Files operations, and Path type. ScoreCard implements the marker and retains the fixed class version identifier. Its String owner and int points are ordinary saved instance fields. Its transient int views belongs to each card but is excluded from the default saved representation.

The constructor assigns the supplied owner and points and sets views to one. The three getters return the corresponding fields; reading a getter does not increment the view count. The static version identifier belongs to the class, so it is not another per-object score value.

### Complete a private round trip

Files.createTempFile creates a new empty file and returns its Path. The prefix and suffix help name it; generated characters distinguish this fixture's file. The .bin suffix does not select the object format.

The outer try creates original as Maya's seven-point card. Files.newOutputStream opens byte output, and ObjectOutputStream wraps it with object-writing support. writeObject receives original. Closing that resource block finishes the writer before the separate input block reads from the same path.

ObjectInputStream wraps byte input. readObject reconstructs a value whose reference is declared as Object. The ScoreCard condition must pass before the cast and getters. A different stored type takes the else branch. The final three print calls compare the saved owner/points and the two separate view counts.

Try-with-resources closes each stream. The outer finally deletes this example's private file even when an operation leaves the try block by throwing. In a conventional Java method, these checked I/O and class-loading exceptions must be handled or declared; the Java notebook kernel supports the shown top-level calls.

These complete notebook fixtures use their class definitions within one kernel session. They do not promise an IJava-generated class identity as a lasting file format. The Super Ghost project supplies compiled sharedCode classes and IOManager declarations; use those actual agreements in MyIOManager.java rather than replacing them with tutorial cards.


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Maya", 7);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Expected output:

```text
Maya: 7
Original views: 1
Restored views: 0
```

The ordinary owner and points fields recover Maya and seven. Constructing original set its views to one. Saving does not erase that original value. The distinct restored card has no saved views value, and default restoration does not replay the ScoreCard constructor, so its transient int begins at zero.

The matching type check allows the cast before any ScoreCard getter is called. A full replay defines the class and creates new objects and a new private file; it does not need an earlier fixture's file.


## Guided Practice

Start with prediction and tracing, then complete a template, change one design choice, and repair a type check. The empty Java work cells are intentionally unfinished until you supply a complete program.


### Predict the changed score

Inspect the complete prediction program below without running it. Predict every printed line for its actual owner and points. Identify saved fields, the restored temporary default, whether the constructor runs again, whether original changes, and the condition needed before casting.


In [ ]:
All predicted output lines:


Saved fields:


Restored views and reason:


Constructor behavior:


Effect on original:


Cast condition:



In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Run the complete prediction program. Keep your prediction above and record every actual line. Explain any correction and why equal saved facts do not require equal temporary counts.


In [ ]:
Actual output:


Confirmed or corrected reasoning:


Saved facts versus temporary counts:



Trace owner, points, views, and serialVersionUID. For each, state whether it is per-object state, whether default serialization saves it as instance state, and its restored value or role. Explain Serializable, the 1L version declaration, readObject, instanceof, and the cast. Include why version agreement is not proof that every class change is compatible.


In [ ]:
owner: per-object / saved / restored:


points: per-object / saved / restored:


views: per-object / saved / restored:


serialVersionUID: per-object / saved / role:


Marker interface:


Version, final, long and L:


Declared Object result, runtime check and cast:


Why a cast does not construct a card:



Identify where the prediction fixture closes the writer and reader and deletes its file. Replay the whole program once, then explain how it supplies its own class, objects, and private file instead of needing an earlier file or kernel.


In [ ]:
Writer closes:


Reader closes:


File cleanup:


Replay output:


Why the replay is self-contained:



<details>
<summary>Show answer</summary>

The saved non-transient instance fields retain owner Iris and points 11. Constructing original sets its transient views field to 1. Default deserialization reconstructs a ScoreCard without replaying that class’s constructor or its instance field initializers, so restored views begins at the int default 0. Reading the restored object does not change original, whose views stays 1.

readObject returns a reference declared as Object. The instanceof ScoreCard check succeeds, and the following cast lets the code use the ScoreCard reader methods. The cast does not create another card or transform its values.

The writer closes before the reader opens; the reader closes before finally deletes the example’s own temporary file. A full replay creates new example objects and a new private temporary file. Its output is checked within that complete round trip; the file is not promised as a format that can be carried into a different kernel session.

Serializable is a marker interface with no required methods. The String and primitive instance fields used here support serialization. The static final long serialVersionUID identifies the class version for part of compatibility checking; it is not ordinary saved per-object state and cannot guarantee that every change to the class is compatible. The L suffix gives 1L the long type, and final prevents reassignment. Object is the declared result type of readObject; the runtime check establishes whether the actual non-null value supports ScoreCard access.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 1
Restored views: 0
```

Common error: Assuming restoration repeats the serializable class’s constructor. Assuming transient means the original field is immediately cleared. Treating a cast as construction of a replacement object.

</details>


### Check a controlled unexpected value

The complete program below creates original but writes the String status. Predict the selected branch and every output line before running. Identify the actual writeObject argument.


In [ ]:
String-file prediction:


Selected branch:


Actual value supplied to writeObject:



In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Run the complete String-file fixture. Record the actual result and explain why the card cast and getter calls are skipped. Distinguish an unexpected type from damaged file bytes.


In [ ]:
Actual output:


Why the cast is skipped:


Type mismatch versus damaged bytes:



Before changing the fixture, predict what happens if only its writeObject argument becomes null. After recording the prediction, edit the earlier complete support program and run that version.


In [ ]:
Null-file prediction:


Predicted branch and reason:



Record the null-file result. Explain why String and null take the same branch for different reasons, why neither permits the card cast, and why constructing original does not determine what was written.


In [ ]:
Actual null-file output:


String rejection reason:


Null rejection reason:


Skipped cast:


What determines the stored value:



<details>
<summary>Show answer</summary>

The file contains the String status, so readObject returns a String reference declared as Object. That value is not compatible with ScoreCard. The instanceof condition is false, the cast and card print statements are skipped, and the else branch prints Unexpected object type. The data is a valid serialized String, but it is the wrong type for this reader.

In the null comparison, readObject returns null; instanceof ScoreCard is again false, so the same branch runs without casting null or calling a card reader.

Constructing original does not force the writer to store it; the actual argument to writeObject determines what this file contains.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

Common error: Assuming the type of original controls the file even when writeObject receives a different argument. Casting before checking the actual restored value. Treating an unexpected stored type as proof that the file bytes are damaged.

**Check case 2.** A null value fails instanceof ScoreCard. The reader follows its mismatch branch without attempting the cast or calling a card method. All file creation, stream closing and cleanup remain inside this complete fixture.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(null);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

</details>


### Complete the known reading pattern

Replace MARKER_INTERFACE, VERSION_ID, RESTORE_OPERATION, and both EXPECTED_TYPE occurrences in this intentionally incomplete template. Use the already taught names and values in their correct roles. Reconstruct the known output, then place your entire completed program in the empty Java work cell and run it.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements MARKER_INTERFACE {
    private static final long serialVersionUID = VERSION_ID;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.RESTORE_OPERATION();
        if (value instanceof EXPECTED_TYPE) {
            ScoreCard restored = (EXPECTED_TYPE) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```


In [ ]:
Four replacements and roles:


Reconstructed expected output:



Record your completed program’s actual output. Explain why the guard and cast must name the same expected type, why the check comes first, and what the fixed version identifier does and does not guarantee.


In [ ]:
Actual output:


Matching guard and cast:


Check-before-cast reason:


Version identifier and limits:



<details>
<summary>Show answer</summary>

Use Serializable in implements, 1L for the version identifier, readObject for the input operation and ScoreCard in both the runtime check and cast. These last two uses must refer to the expected card type. The check belongs before the cast so an incompatible value takes the alternate branch.

The saved non-transient instance fields retain owner Iris and points 11. Constructing original sets its transient views field to 1. Default deserialization reconstructs a ScoreCard without replaying that class’s constructor or its instance field initializers, so restored views begins at the int default 0. Reading the restored object does not change original, whose views stays 1.

readObject returns a reference declared as Object. The instanceof ScoreCard check succeeds, and the following cast lets the code use the ScoreCard reader methods. The cast does not create another card or transform its values.

The writer closes before the reader opens; the reader closes before finally deletes the example’s own temporary file.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 1
Restored views: 0
```

Common error: Using different target types in the check and cast. Replacing the object input operation with a primitive read from the previous lesson. Treating the marker interface as a replacement for the class’s own reader methods.

</details>


### Change temporary initialization

Recall the unchanged Iris fixture’s output, rerunning the earlier prediction program if needed. The starter below preserves that baseline. Predict every line when only the constructor’s views assignment changes from one to five and transient stays on views. Then run the unchanged starter once, edit that assignment, and run the complete changed program.


In [ ]:
Unchanged baseline output:


Prediction: views five with transient retained:



In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}


Record the result with original views initialized to five and transient retained. Explain the original and restored counts using normal construction and default restoration.


In [ ]:
Actual output:


Why original has this count:


Why restored has this count:



Keep a copy of the transient version. Predict the separate comparison that also removes transient from views while retaining the constructor assignment of five. After predicting, make that declaration change and run the entire complete fixture.


In [ ]:
Prediction: views five without transient:


Which value now belongs in saved state:



Record the non-transient comparison. Explain why the declaration changes the restored value. Choose the declaration appropriate for a temporary view count versus a count the design requires saving.


In [ ]:
Actual output:


Why the value survives:


Temporary-state design:


Lasting-state design:



<details>
<summary>Show answer</summary>

Changing the constructor sets original views to 5, but the field remains transient in the first modification. The restored count still begins at 0 because default deserialization does not replay this serializable class’s constructor assignment. Owner Iris and points 11 remain saved.

In the separate comparison, removing transient makes views an ordinary non-static instance field, so the original value 5 is saved and restored. The result then has Original views: 5 and Restored views: 5. Choose the declaration from the intended lifetime of the data; do not discard a needed value merely to force a desired output.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 5;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 5
Restored views: 0
```

Common error: Expecting the changed constructor assignment to initialize the transient field during restoration. Assuming removing transient changes the original object’s already assigned value. Excluding a field that the design requires restoring.

**Additional test: `Constructor sets views 5; views is no longer transient`.** The complete fresh fixture defines views as an ordinary instance field before creating and writing its object. Its value 5 is therefore restored with owner and points. The earlier transient fixture remains a separate complete comparison.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 5;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 5
Restored views: 5
```

</details>


### Repair a mismatched guard

This intentionally faulty complete fixture writes a String but tests instanceof String before a ScoreCard cast. Predict the first failing operation and whether any card lines print. Copy it into the empty work cell and run this labeled diagnostic after predicting.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof String) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```


In [ ]:
String-file prediction:


First failing operation:


Any earlier named card output:


Actual diagnostic observed:



Keep the wrong String guard for a second controlled diagnostic. Predict what it does when writeObject receives original instead. Then change only that argument in the earlier work cell, run the complete fixture, and record the result before repairing the guard.


In [ ]:
Prediction for actual ScoreCard under wrong guard:


Actual output:


Why this guard rejects the intended card:



Repair the condition to check the type required by the unchanged ScoreCard cast. Predict the corrected String-file result, run that complete fixture, and record the output. Then predict and run the corrected original-card version. Explain why accepting the right type and rejecting the wrong type are both needed.


In [ ]:
Repaired condition:


Corrected String prediction:


Corrected String actual output:


Corrected card prediction:


Corrected card actual output:


Why both tests matter:



<details>
<summary>Show answer</summary>

With the String-file draft, the wrong instanceof String condition succeeds, but the following ScoreCard cast cannot access that String as a ScoreCard. It raises ClassCastException before the first card print statement, so no named card lines are printed.

With writeObject(original) and the same wrong String guard, the actual ScoreCard fails that condition and the program instead prints Unexpected object type. The guard is wrong for both cases.

Repair it to instanceof ScoreCard before the unchanged ScoreCard cast. The corrected String-file program safely prints `Unexpected object type.`; the corrected card-file program prints Iris: 11, Original views: 1 and Restored views: 0.

Both complete fixtures close their streams and delete their own files. A type check only helps when it checks the type required by the operation that follows.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

Common error: Checking for String before casting to ScoreCard. Moving the cast outside the guarded branch. Considering a reader correct merely because it rejects every input.

**Additional test: `After repairing the guard, write original ScoreCard instead of String`.** The correct guard accepts the restored ScoreCard and permits its cast and reader methods. This complements rejecting the String; an always-false condition could not pass both tests.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class ScoreCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String owner;
    private int points;
    private transient int views;
    public ScoreCard(String owner, int points) {
        this.owner = owner;
        this.points = points;
        this.views = 1;
    }
    public String getOwner() { return owner; }
    public int getPoints() { return points; }
    public int getViews() { return views; }
}
Path file = Files.createTempFile("score-card-", ".bin");
try {
    ScoreCard original = new ScoreCard("Iris", 11);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof ScoreCard) {
            ScoreCard restored = (ScoreCard) value;
            System.out.println(restored.getOwner() + ": " + restored.getPoints());
            System.out.println("Original views: " + original.getViews());
            System.out.println("Restored views: " + restored.getViews());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Iris: 11
Original views: 1
Restored views: 0
```

</details>


## Independent Practice

### Restore lasting equipment information

Use the same saved-versus-temporary reasoning in a different class. Supply a complete fixture so another student can run it from a fresh Java kernel.


Define EquipmentCard implementing Serializable with private static final long serialVersionUID = 1L. Give it private String item, int available, and transient boolean selected fields. Its constructor accepts item and available, assigns them, and sets selected true. Supply getItem, getAvailable, and isSelected getters.

Create a camera card with three available. Save it to a new private file, read the result as Object, check instanceof EquipmentCard, and cast only inside that branch. Report the item and available count, then Original selected and Restored selected on separate labeled lines. Use Unexpected object type. for a mismatch. Include every import and setup step, close both streams, and delete the fixture’s file.

Plan the field choices and predict the report before writing and running your complete program in the work cell.


In [ ]:
Saved fields and meanings:


Temporary field and intended lifetime:


Baseline predicted output:


Plan for check, cast, resources and cleanup:



Record every line from your baseline EquipmentCard program. Explain the saved values, original selection, and restored default. Identify the matching guard/cast and the closing and cleanup points. Keep the baseline for the following comparisons.


In [ ]:
Baseline actual output:


Saved values and temporary default:


Guard and cast:


Stream closing:


File cleanup:



Test a different item: change only camera to projector, keeping three available. Write the prediction below before running. Then run the complete variant with a fresh private file and add its actual output and explanation in the separate labeled space.


In [ ]:
Projector-three prediction BEFORE run:


Projector-three actual output AFTER run:


Explanation after run:



Return to camera and change available to zero. Zero is a valid available count. Predict every output line first; then run this complete variant and record its output and explanation.


In [ ]:
Camera-zero prediction BEFORE run:


Camera-zero actual output AFTER run:


Explanation after run:



Restore the baseline original camera card with three available, but change the writeObject argument to the String status. Keep the EquipmentCard check and cast. Predict first, then run the complete fixture and explain the actual result.


In [ ]:
String-status prediction BEFORE run:


String-status actual output AFTER run:


Explanation after run:



Compare all four EquipmentCard cases. Identify the case that must skip the cast and explain why. Explain saved-field values versus transient defaults, whether any read changes original, and how each full test closes streams and removes only its own file.


In [ ]:
Case that skips the cast and why:


Saved fields versus temporary defaults:


Effect on original:


Closing and cleanup in all four cases:



<details>
<summary>Show answer</summary>

EquipmentCard implements the Serializable marker interface and declares the fixed static final long version identifier. Its String item and int available are ordinary instance fields. They restore camera and 3. The constructor sets original selected to true; selected is transient, so default deserialization does not save it or replay that constructor assignment and restored selected begins at false.

The readObject result is held as Object, checked with instanceof EquipmentCard and cast only inside the matching branch. Reading this restored object does not clear selected in original. Both streams close and finally removes the complete example’s private temporary file.

The different item and zero availability are ordinary saved-field values; changing them does not change the transient default.

The controlled String test stores a valid serialized String instead of the card. It reaches the mismatch branch and prints `Unexpected object type.` without casting or calling EquipmentCard readers. This test checks the actual stored type rather than the class of an unrelated original variable. All tests use only files created within their own complete programs.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("camera", 3);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
camera: 3
Original selected: true
Restored selected: false
```

Common error: Leaving selected as ordinary saved state despite the temporary-field requirement. Assuming the constructor is replayed to set the restored selection true. Depending on a prior notebook class or temporary file.

**Additional test: Different item projector, available 3.** The different String is saved and restored; the transient boolean still begins false in the reconstructed card.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("projector", 3);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
projector: 3
Original selected: true
Restored selected: false
```

**Additional test: Camera with zero available.** Zero is a valid saved int value and remains zero after restoration. It does not change the separate transient selection rule.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("camera", 0);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(original);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
camera: 0
Original selected: true
Restored selected: false
```

**Additional test: Baseline original card exists, but writeObject receives String status.** The actual saved value is a String. The EquipmentCard guard fails and the cast is skipped, regardless of the separately created original card.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class EquipmentCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String item;
    private int available;
    private transient boolean selected;
    public EquipmentCard(String item, int available) {
        this.item = item;
        this.available = available;
        this.selected = true;
    }
    public String getItem() { return item; }
    public int getAvailable() { return available; }
    public boolean isSelected() { return selected; }
}
Path file = Files.createTempFile("equipment-card-", ".bin");
try {
    EquipmentCard original = new EquipmentCard("camera", 3);
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof EquipmentCard) {
            EquipmentCard restored = (EquipmentCard) value;
            System.out.println(restored.getItem() + ": " + restored.getAvailable());
            System.out.println("Original selected: " + original.isSelected());
            System.out.println("Restored selected: " + restored.isSelected());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

</details>


## Summary

Ordinary supported instance fields carry the saved facts. A transient field remains part of each object but is excluded from default serialization. In these ordinary Serializable classes, restoration does not replay the serializable constructor or instance initializers; transient int and boolean fields begin at zero and false.

Saving and restoring do not clear the original object's temporary fields. Check the restored type before casting, and choose excluded fields from the intended lifetime of the data rather than from an output you want to force.


Close the answers. Explain why a restored object’s saved fields and temporary fields can have different relationships to the original. Include constructor behavior and the type’s default value.


In [ ]:
My explanation from memory:



<details>
<summary>Show answer</summary>

The saved owner and point fields recover their stored values. A transient view count has no stored value in this default representation. Restoration does not repeat the ScoreCard constructor, so the restored int begins at zero. The original card keeps the value assigned during its own construction.

A temporary selection follows the same reasoning with a boolean default of false. If the application needs a lasting business value, that requirement should guide its field design. An instanceof check protects the following card cast only when both name the required type.

</details>


## Reflection

Connect the saving decision to a class in a field that interests you. Next, JavaFX will present object state through a graphical interface in Workspace Desktop.


Choose an object in a field that interests you. Identify lasting information and temporary interface state, choose what should survive restoration, and explain whether each temporary field’s default is suitable. Describe how the reader should respond to a wrong restored type.


In [ ]:
Object and purpose:


Lasting fields:


Temporary fields and suitable restored values:


Wrong-type response:



## Supplemental Reading

- [Serializable and class compatibility](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/Serializable.html) defines the marker interface and version identifier.
- [ObjectOutputStream](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/ObjectOutputStream.html) explains which object fields are written.
- [ObjectInputStream](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/ObjectInputStream.html) explains restoration, result types, and exceptions.
- [ClassCastException](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/ClassCastException.html) describes an incompatible reference cast.
- [Serialization architecture](https://docs.oracle.com/en/java/javase/21/docs/specs/serialization/serial-arch.html) describes saved state and transient-field behavior.
